# FinBERT + LDA + LSTM — pooled-outlet option + lag testing 


In [ ]:
!pip install -q transformers torch gensim scikit-learn statsmodels pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 52.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/brexit_covid_datasets"
print("Reading from:", DATA_DIR)

Mounted at /content/drive
Reading from: /content/drive/MyDrive/brexit_covid_datasets


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

ECON_INDICATOR_PATH = f"{DATA_DIR}/economic_indicator.csv"
OUTLETS = ["bbc", "dailymail", "guardian"]
TOPICS = ["brexit", "covid", "energy_crisis"]
ENTITIES = OUTLETS + ["pooled"]   # "pooled" = all three outlets combined, alongside them

AGG_FREQ = "MS"
LSTM_LOOKBACK = 3
TEST_FRACTION = 0.2
RANDOM_STATE = 42

LAGS_MONTHS = [0, 1, 2, 3, 6]

RECESSION_WINDOWS = [
    {"name": "2020 COVID recession", "start": pd.Timestamp("2020-01-01"), "end": pd.Timestamp("2020-06-30")},
    {"name": "2023 H2 recession",    "start": pd.Timestamp("2023-07-01"), "end": pd.Timestamp("2023-12-31")},
]
LEADUP_MONTHS = 6
CALM_COMPARISON_MONTHS = 6

## Load articles per (outlet, topic)

In [ ]:
def load_outlet_topic(outlet, topic):
    if topic == "energy_crisis":
        path = f"{DATA_DIR}/{outlet}_energy_crisis_TEST.csv"
    else:
        path = f"{DATA_DIR}/{outlet}_brexit_covid_text_FINAL.csv"
    df = pd.read_csv(path)
    df = df[df["topic"] == topic].copy()
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True).dt.tz_localize(None)
    df = df.dropna(subset=["publish_date", "text"]).sort_values("publish_date").reset_index(drop=True)
    return df

datasets = {}
for outlet in OUTLETS:
    for topic in TOPICS:
        key = (outlet, topic)
        try:
            df = load_outlet_topic(outlet, topic)
            datasets[key] = df
            print(f"{outlet}/{topic}: {len(df)} articles, {df['publish_date'].min()} to {df['publish_date'].max()}")
        except FileNotFoundError:
            print(f"{outlet}/{topic}: FILE NOT FOUND")

bbc/brexit: 794 articles, 2012-06-17 00:00:00 to 2026-07-21 00:00:00
bbc/covid: 790 articles, 2020-01-10 00:00:00 to 2026-08-08 00:00:00
bbc/energy_crisis: 800 articles, 2021-08-06 00:00:00 to 2026-08-12 00:00:00
dailymail/brexit: 799 articles, 2012-01-28 00:00:00 to 2026-03-16 00:00:00
dailymail/covid: 798 articles, 2020-01-05 00:00:00 to 2026-03-16 00:00:00
dailymail/energy_crisis: 799 articles, 2021-08-07 00:00:00 to 2026-03-16 00:00:00
guardian/brexit: 800 articles, 2012-02-10 21:00:04 to 2026-08-04 04:00:12
guardian/covid: 800 articles, 2020-03-23 20:41:20 to 2026-08-08 20:03:34
guardian/energy_crisis: 800 articles, 2021-08-03 05:31:14 to 2026-08-13 14:54:03


## GDP growth target

In [ ]:
econ = pd.read_csv(ECON_INDICATOR_PATH)
econ["date"] = pd.to_datetime(econ["date"])
econ = econ.set_index("date").resample(AGG_FREQ).mean().rename(columns={"value": "econ_value"})
econ["growth"] = econ["econ_value"].pct_change() * 100
print(f"GDP growth series: {econ.index.min()} to {econ.index.max()}, {len(econ)} months")

GDP growth series: 1997-01-01 00:00:00 to 2026-05-01 00:00:00, 353 months


## Model 1 — FinBERT sentiment (whole-article, same as the main pipeline)

In [ ]:
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    truncation=True,
    max_length=512,
    model_kwargs={"use_safetensors": False},
)

def finbert_signed_scores(texts, batch_size=16):
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = [t[:2000] if isinstance(t, str) else "" for t in texts[i:i+batch_size]]
        results = finbert(batch)
        for r in results:
            label, score = r["label"].lower(), r["score"]
            scores.append(score if label == "positive" else (-score if label == "negative" else 0.0))
    return scores

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Model 2 — LDA topic modeling (same as the main pipeline)

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

def fit_lda(texts, num_topics=5, passes=10):
    tokenized = [[w for w in simple_preprocess(str(t)) if w not in STOPWORDS and len(w) > 2] for t in texts]
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=2, no_above=0.5)
    corpus = [dictionary.doc2bow(t) for t in tokenized]
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=passes, random_state=RANDOM_STATE)
    return lda, dictionary, corpus

def lda_topic_scores(lda, corpus):
    scores = []
    for bow in corpus:
        dist = dict(lda.get_document_topics(bow, minimum_probability=0))
        scores.append(max(dist.values()) if dist else 0.0)
    return scores

## Models — Ridge, multivariate LSTM, naive + ARIMA baselines

In [ ]:
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.arima.model import ARIMA

RIDGE_ALPHA_GRID = (0.1, 0.3, 1.0, 3.0, 10.0, 30.0)

def tune_ridge_alpha(X_train, y_train, alphas=RIDGE_ALPHA_GRID, min_months_for_cv=24):
    n = len(y_train)
    if n < min_months_for_cv:
        return 1.0, {1.0: None}
    n_splits = min(5, max(2, n // 12))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = {}
    for alpha in alphas:
        fold_rmses = []
        for tr_idx, val_idx in tscv.split(X_train):
            if len(tr_idx) < 3 or len(val_idx) < 1:
                continue
            m = Ridge(alpha=alpha).fit(X_train[tr_idx], y_train[tr_idx])
            pred = m.predict(X_train[val_idx])
            fold_rmses.append(mean_squared_error(y_train[val_idx], pred) ** 0.5)
        scores[alpha] = np.mean(fold_rmses) if fold_rmses else np.inf
    best_alpha = min(scores, key=scores.get)
    return best_alpha, scores

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    direction_match = np.sign(y_true) == np.sign(y_pred)
    return {
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan"),
        "directional_accuracy": direction_match.mean(),
    }

class SentimentLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def train_lstm(X_train, y_train, input_size, epochs=150, lr=0.01, hidden_size=16, patience=15, val_fraction=0.2):
    torch.manual_seed(RANDOM_STATE)
    n = len(y_train)
    n_val = max(1, int(n * val_fraction)) if n >= 10 else 0
    if n_val > 0:
        X_fit, y_fit = X_train[:-n_val], y_train[:-n_val]
        X_val, y_val = X_train[-n_val:], y_train[-n_val:]
    else:
        X_fit, y_fit = X_train, y_train
        X_val, y_val = None, None

    model = SentimentLSTM(input_size=input_size, hidden_size=hidden_size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    Xt = torch.tensor(X_fit)
    yt = torch.tensor(y_fit)

    best_val_loss, best_state, epochs_without_improve, stopped_at = float("inf"), None, 0, epochs
    train_loss_history, val_loss_history = [], []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            train_loss_history.append(loss_fn(model(Xt), yt).item())

        if X_val is not None:
            with torch.no_grad():
                val_loss = loss_fn(model(torch.tensor(X_val)), torch.tensor(y_val)).item()
            val_loss_history.append(val_loss)
            if val_loss < best_val_loss:
                best_val_loss, epochs_without_improve = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                epochs_without_improve += 1
                if epochs_without_improve >= patience:
                    stopped_at = epoch + 1
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    model._stopped_at_epoch = stopped_at
    model._train_loss_history = train_loss_history
    model._val_loss_history = val_loss_history
    model._train_var = float(np.var(y_fit))
    model._val_var = float(np.var(y_val)) if X_val is not None else None
    return model

def predict_lstm(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).numpy()

def naive_persistence_forecast(y_train, n_test):
    return np.full(n_test, y_train[-1])

def select_arima_order(y_train, candidate_orders=((1, 0, 0), (1, 0, 1))):
    best_order, best_aic = (1, 0, 0), float("inf")
    for order in candidate_orders:
        try:
            fit = ARIMA(y_train, order=order).fit()
            if fit.aic < best_aic:
                best_aic, best_order = fit.aic, order
        except Exception:
            continue
    return best_order

def fit_arima_baseline(y_train, n_test, order=None):
    if order is None:
        order = select_arima_order(y_train)
    try:
        model = ARIMA(y_train, order=order).fit()
        return np.asarray(model.forecast(steps=n_test)), order
    except Exception as e:
        print(f"ARIMA failed ({e}) — falling back to naive baseline")
        return naive_persistence_forecast(y_train, n_test), order

## Score FinBERT + LDA per (entity, topic), monthly 

In [ ]:
FEATURE_COLS = [f"{t}_{m}" for t in TOPICS for m in ("finbert", "lda")]  # 6 columns total
score_cache = {}       # (entity, topic) -> DataFrame with monthly finbert/lda columns
wide_cache = {}         # entity -> full wide monthly feature table (no growth/lag yet)

def gather_articles(entity, topic):
    outlets_to_use = OUTLETS if entity == "pooled" else [entity]
    frames = [datasets[(o, topic)] for o in outlets_to_use if (o, topic) in datasets and len(datasets[(o, topic)])]
    if not frames:
        return pd.DataFrame(columns=["publish_date", "text"])
    return pd.concat(frames, ignore_index=True)

def score_topic_monthly(entity, topic):
    key = (entity, topic)
    if key in score_cache:
        return score_cache[key]

    df = gather_articles(entity, topic)
    if len(df) < 20:
        score_cache[key] = None
        return None

    texts = df["text"].tolist()
    finbert_scores = finbert_signed_scores(texts)

    lda, dictionary, corpus = fit_lda(texts)
    lda_scores = lda_topic_scores(lda, corpus)

    scored = pd.DataFrame({
        "publish_date": df["publish_date"].values,
        "finbert": finbert_scores,
        "lda": lda_scores,
    })
    monthly = scored.set_index("publish_date").resample(AGG_FREQ).mean()
    monthly.columns = [f"{topic}_finbert", f"{topic}_lda"]
    score_cache[key] = monthly
    return monthly

def build_wide_features(entity):
    if entity in wide_cache:
        return wide_cache[entity]
    parts = []
    for topic in TOPICS:
        monthly = score_topic_monthly(entity, topic)
        if monthly is not None:
            parts.append(monthly)
    if not parts:
        wide_cache[entity] = pd.DataFrame(columns=FEATURE_COLS)
        return wide_cache[entity]

    wide = pd.concat(parts, axis=1)   # outer join by default -> union of months
    wide = wide.reindex(columns=FEATURE_COLS)  # keep a consistent column order/set
    wide = wide.ffill().fillna(0.0)   # carry sentiment forward through gaps; neutral before a topic starts
    wide_cache[entity] = wide
    return wide

def build_lagged_table(entity, lag_months):
    """Row m = features from month m, paired with GDP growth at month (m + lag_months)."""
    wide = build_wide_features(entity)
    rows = []
    for m, feat_row in wide.iterrows():
        target_date = m + pd.DateOffset(months=lag_months)
        if target_date in econ.index and pd.notna(econ.loc[target_date, "growth"]):
            row = {"text_month": m, "target_date": target_date, "growth": econ.loc[target_date, "growth"]}
            row.update(feat_row.to_dict())
            rows.append(row)
    return pd.DataFrame(rows).sort_values("text_month").reset_index(drop=True)

## Main loop — per entity (bbc/dailymail/guardian/pooled), per lag: Ridge + multivariate LSTM + baselines

In [ ]:
results = []

for entity in ENTITIES:
    print(f"\n=== Scoring {entity} (FinBERT + LDA, all 3 topics) ===")
    wide = build_wide_features(entity)
    print(f"{entity}: {len(wide)} months of feature coverage")

    for lag in LAGS_MONTHS:
        aligned = build_lagged_table(entity, lag)
        n = len(aligned)
        print(f"\n--- {entity} / lag={lag}mo ({n} aligned months) ---")
        if n < 20:
            print("Too few months — skipping.")
            continue

        split_idx = int(n * (1 - TEST_FRACTION))
        X = aligned[FEATURE_COLS].values
        y = aligned["growth"].values
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        alpha, _ = tune_ridge_alpha(X_train, y_train)
        ridge_pred = Ridge(alpha=alpha).fit(X_train, y_train).predict(X_test)
        results.append({"entity": entity, "lag_months": lag, "model": "Ridge (FinBERT+LDA)",
                         **evaluate(y_test, ridge_pred), "n_months": n, "tuned_param": f"alpha={alpha}"})

        if n - LSTM_LOOKBACK < 12:
            print("Not enough months for LSTM lookback — skipping LSTM.")
        else:
            Xs, ys = [], []
            for i in range(len(X) - LSTM_LOOKBACK):
                Xs.append(X[i:i + LSTM_LOOKBACK])
                ys.append(y[i + LSTM_LOOKBACK])
            Xs, ys = np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)
            split_seq = int(len(Xs) * (1 - TEST_FRACTION))

            y_scaler = StandardScaler()
            y_train_scaled = y_scaler.fit_transform(ys[:split_seq].reshape(-1, 1)).flatten()
            lstm_model = train_lstm(Xs[:split_seq], y_train_scaled, input_size=len(FEATURE_COLS))
            pred_scaled = predict_lstm(lstm_model, Xs[split_seq:])
            pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            results.append({"entity": entity, "lag_months": lag, "model": "Multivariate-LSTM (FinBERT+LDA)",
                             **evaluate(ys[split_seq:], pred), "n_months": len(Xs),
                             "tuned_param": f"stopped_epoch={lstm_model._stopped_at_epoch}"})

        naive_pred = naive_persistence_forecast(y_train, len(y_test))
        results.append({"entity": entity, "lag_months": lag, "model": "Naive (no-change)",
                         **evaluate(y_test, naive_pred), "n_months": n, "tuned_param": ""})

        arima_pred, arima_order = fit_arima_baseline(y_train, len(y_test))
        results.append({"entity": entity, "lag_months": lag, "model": "ARIMA(auto)",
                         **evaluate(y_test, arima_pred), "n_months": n, "tuned_param": f"order={arima_order}"})

results_df = pd.DataFrame(results)
print("\n\n=== FINAL RESULTS (FinBERT+LDA, all lags, pooled + per-outlet) ===")
print(results_df)
results_df.to_csv(f"{DATA_DIR}/model_results_sentlda_lags_pooled_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/model_results_sentlda_lags_pooled_TEST.csv")


=== Scoring bbc (FinBERT + LDA, all 3 topics) ===


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


bbc: 171 months of feature coverage

--- bbc / lag=0mo (168 aligned months) ---

--- bbc / lag=1mo (167 aligned months) ---

--- bbc / lag=2mo (166 aligned months) ---

--- bbc / lag=3mo (165 aligned months) ---

--- bbc / lag=6mo (162 aligned months) ---

=== Scoring dailymail (FinBERT + LDA, all 3 topics) ===
dailymail: 171 months of feature coverage

--- dailymail / lag=0mo (171 aligned months) ---

--- dailymail / lag=1mo (171 aligned months) ---

--- dailymail / lag=2mo (171 aligned months) ---

--- dailymail / lag=3mo (170 aligned months) ---

--- dailymail / lag=6mo (167 aligned months) ---

=== Scoring guardian (FinBERT + LDA, all 3 topics) ===
guardian: 175 months of feature coverage

--- guardian / lag=0mo (172 aligned months) ---

--- guardian / lag=1mo (171 aligned months) ---

--- guardian / lag=2mo (170 aligned months) ---

--- guardian / lag=3mo (169 aligned months) ---

--- guardian / lag=6mo (166 aligned months) ---

=== Scoring pooled (FinBERT + LDA, all 3 topics) ===

## Recession-specific test — both windows, per entity, per lag

Same target-date-based train/test split as the term-count notebook: train only ever
uses rows whose TARGET date (text month + lag) falls before the recession starts, so
a lagged model can never see the recession's own outcome during training.

In [ ]:
recession_results = []

for entity in ENTITIES:
    for lag in LAGS_MONTHS:
        aligned = build_lagged_table(entity, lag)
        if len(aligned) < 20:
            continue

        for window in RECESSION_WINDOWS:
            rec_start, rec_end, label = window["start"], window["end"], window["name"]
            leadup_start = rec_start - pd.DateOffset(months=LEADUP_MONTHS)
            calm_end = rec_end + pd.DateOffset(months=CALM_COMPARISON_MONTHS)

            train = aligned[aligned["target_date"] < rec_start]
            test_leadup = aligned[(aligned["target_date"] >= leadup_start) & (aligned["target_date"] < rec_start)]
            test_recession = aligned[(aligned["target_date"] >= rec_start) & (aligned["target_date"] <= rec_end)]
            test_calm = aligned[(aligned["target_date"] > rec_end) & (aligned["target_date"] <= calm_end)]

            tag = f"{entity}/lag={lag}mo [{label}]"
            if len(train) < 24 or len(test_recession) < 2:
                print(f"{tag}: {len(train)} training months — not enough data, skipping")
                continue

            X_train = train[FEATURE_COLS].values
            y_train = train["growth"].values
            alpha_r, _ = tune_ridge_alpha(X_train, y_train)
            m = Ridge(alpha=alpha_r).fit(X_train, y_train)
            print(f"{tag}: {len(train)} training months, tuned alpha={alpha_r}")

            for period_name, test_set in [
                ("lead-up (pre-recession)", test_leadup),
                ("recession", test_recession),
                ("calm (post-recession)", test_calm),
            ]:
                if len(test_set) < 2:
                    continue
                X_test = test_set[FEATURE_COLS].values
                pred = m.predict(X_test)
                recession_results.append({
                    "entity": entity, "lag_months": lag, "recession": label, "period": period_name,
                    **evaluate(test_set["growth"].values, pred),
                    "n_months": len(test_set), "tuned_param": f"alpha={alpha_r}",
                })

recession_df = pd.DataFrame(recession_results)
print("\n\n=== LEAD-UP vs RECESSION vs CALM COMPARISON (FinBERT+LDA, all lags) ===")
print(recession_df)
recession_df.to_csv(f"{DATA_DIR}/recession_specific_results_sentlda_lags_pooled_TEST.csv", index=False)
print(f"\nSaved to {DATA_DIR}/recession_specific_results_sentlda_lags_pooled_TEST.csv")

bbc/lag=0mo [2020 COVID recession]: 91 training months, tuned alpha=30.0
bbc/lag=0mo [2023 H2 recession]: 133 training months, tuned alpha=30.0
bbc/lag=1mo [2020 COVID recession]: 90 training months, tuned alpha=10.0
bbc/lag=1mo [2023 H2 recession]: 132 training months, tuned alpha=30.0
bbc/lag=2mo [2020 COVID recession]: 89 training months, tuned alpha=1.0
bbc/lag=2mo [2023 H2 recession]: 131 training months, tuned alpha=30.0
bbc/lag=3mo [2020 COVID recession]: 88 training months, tuned alpha=30.0
bbc/lag=3mo [2023 H2 recession]: 130 training months, tuned alpha=30.0
bbc/lag=6mo [2020 COVID recession]: 85 training months, tuned alpha=10.0
bbc/lag=6mo [2023 H2 recession]: 127 training months, tuned alpha=30.0
dailymail/lag=0mo [2020 COVID recession]: 96 training months, tuned alpha=3.0
dailymail/lag=0mo [2023 H2 recession]: 138 training months, tuned alpha=30.0
dailymail/lag=1mo [2020 COVID recession]: 95 training months, tuned alpha=30.0
dailymail/lag=1mo [2023 H2 recession]: 137 trai

## Clear, separated results tables

In [ ]:
print("=== MAIN RESULTS, by entity / lag ===\n")
for (entity, lag), sub in results_df.groupby(["entity", "lag_months"]):
    print(f"--- {entity} / lag={lag}mo ---")
    print(sub[["model", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
    print()

if len(recession_df):
    print("\n=== RECESSION-SPECIFIC RESULTS, by entity / lag / recession ===\n")
    for (entity, lag, rec), sub in recession_df.groupby(["entity", "lag_months", "recession"]):
        print(f"--- {entity} / lag={lag}mo / {rec} ---")
        print(sub[["period", "rmse", "mae", "r2", "directional_accuracy", "n_months", "tuned_param"]].to_string(index=False))
        print()
else:
    print("\nNo entity/lag combination had 24+ pre-recession training months for the recession-specific test.")

print("\n=== Quick scan: best directional accuracy per entity (main loop, across all lags/models) ===\n")
if len(results_df):
    best = results_df.loc[results_df.groupby("entity")["directional_accuracy"].idxmax()]
    print(best[["entity", "lag_months", "model", "directional_accuracy", "r2", "n_months"]].to_string(index=False))

=== MAIN RESULTS, by entity / lag ===

--- bbc / lag=0mo ---
                          model     rmse      mae        r2  directional_accuracy  n_months      tuned_param
            Ridge (FinBERT+LDA) 0.234520 0.196880 -0.044309              0.529412       168       alpha=30.0
Multivariate-LSTM (FinBERT+LDA) 0.230755 0.190542 -0.002098              0.545455       165 stopped_epoch=17
              Naive (no-change) 0.539283 0.488134 -4.522067              0.294118       168                 
                    ARIMA(auto) 0.238077 0.202504 -0.076219              0.558824       168  order=(1, 0, 1)

--- bbc / lag=1mo ---
                          model     rmse      mae        r2  directional_accuracy  n_months      tuned_param
            Ridge (FinBERT+LDA) 0.228681 0.192239  0.007046              0.529412       167       alpha=30.0
Multivariate-LSTM (FinBERT+LDA) 0.236663 0.199674 -0.054071              0.545455       164 stopped_epoch=18
              Naive (no-change) 0.539283 0.4